In [1]:
!pip install nltk
import pandas as pd
import numpy as np
from pathlib import Path

df = pd.read_csv('processed/reviews_merged.csv')
top250 = pd.read_csv('raw/imdb_top250.csv')
df['title'] = df['imdb_id'].map(dict(zip(top250['imdb_id'], top250['title'])))

print(f'Yorum: {len(df):,} | Film: {df["imdb_id"].nunique()}')
print(df['language'].value_counts().to_string())


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Yorum: 382,977 | Film: 229
language
en    191562
tr    191415


In [2]:
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords', quiet=True)

TR_EXTRA = {
    'bir', 'iki', 'üç', 'tüm', 'olan', 'oldu', 'olduğu', 'olmuş', 'olmak',
    'yani', 'işte', 'şey', 'şu', 'bunu', 'şunu', 'onu', 'bana', 'sana', 'ona',
    'bizi', 'sizi', 'onları', 'benim', 'senin', 'onun', 'bizim', 'sizin', 'onların',
    'kendi', 'kendisi', 'kendine', 'kendini', 'kendim', 'kendin',
    'sonra', 'önce', 'şimdi', 'bugün', 'yarın', 'dün', 'hala', 'henüz', 'artık',
    'sadece', 'yalnız', 'yalnızca', 'ancak', 'fakat', 'lakin', 'oysa', 'halbuki',
    'çünkü', 'zira', 'madem', 'mademki', 'eğer', 'şayet',
    'mı', 'mi', 'mu', 'mü', 'ki', 'ya', 'yada', 'veya', 'veyahut',
    'pek', 'hiçbir', 'birçok', 'bazen', 'genellikle', 'genelde', 'özellikle',
    'tamamen', 'tam', 'tabi', 'tabii', 'evet', 'hayır', 'belki',
    'aynı', 'farklı', 'başka', 'diğer', 'falan', 'filan', 'felan',
    'yine', 'gene', 'tekrar', 'kez', 'defa', 'kere',
    'gerek', 'gerekir', 'gerekiyor', 'lazım',
    'var', 'yok', 'olur', 'olmaz', 'olabilir',
    'biraz', 'birazcık', 'azıcık',
    'olarak', 'rağmen', 'göre', 'kadar', 'doğru', 'beri',
    'üzerine', 'üzerinde', 'altına', 'altında', 'içine', 'içinde',
    'şöyle', 'böyle', 'öyle', 'nasıl', 'niçin', 'neden', 'nerede', 'nereye',
    'kim', 'kime', 'kimi', 'kimin', 'hangi',
    've', 'ile', 'de', 'da',
    'ol', 'et', 'yap', 'gel', 'git', 'al', 'ver', 'çık', 'bı', 'bır',
    'bun', 'şun', 'ben', 'sen', 'biz', 'siz', 'tane',
    'bi', 'cok', 'guzel', 'fılm', 'fil', 'iyi', 'kötü',
    'nin', 'nun', 'nı', 'ye', 'ba', 'si', 'yı', 'yu', 'yü',
    'haha', 'ha', 'hah', 'hihi', 'hehe',
    'teşekkür', 'tebrik', 'arkadaş', 'merhaba', 'selam',
    'beğen', 'saygı', 'eleştir', 'eleştiri', 'yorum',
    'sene', 'yıl', 'gün', 'ay', 'hafta',
    'hatırla', 'unut', 'kaç', 'oku', 'yaz', 'anla',
    'tanrı', 'allah', 'düşünce', 'fikir',
    'uzer', 'üzer', 'edil', 'olun', 'görün',
    'demek', 'diye', 'gibi',
}
TR_DOMAIN = {
    'film', 'filmi', 'filmin', 'filme', 'filmler', 'filmleri',
    'sinema', 'sinemanın', 'sinemada',
    'izle', 'izledim', 'izlemek', 'izleyen', 'izleyici', 'izleme',
    'oyuncu', 'oyuncusu', 'oyuncular', 'oyunculuk',
    'yönetmen', 'yönetmeni',
    'sahne', 'sahnesi', 'sahneler',
    'yapım', 'yapımı',
    'senaryo', 'senaryosu',
    'karakter', 'karakteri', 'karakterler',
    'rol', 'rolü', 'roller',
    'bence', 'spoiler', 'bkz',
}
EN_DOMAIN = {
    'film', 'films', 'movie', 'movies', 'cinema', 'cinematic',
    'watch', 'watching', 'watched', 'viewer', 'viewing',
    'actor', 'actress', 'actors', 'actresses', 'acting', 'cast',
    'director', 'directing', 'directed',
    'scene', 'scenes', 'shot', 'shots',
    'character', 'characters', 'role', 'roles',
    'story', 'plot', 'screenplay', 'script',
    'see', 'seen', 'seeing', 'saw',
    'really', 'pretty', 'quite', 'just', 'also', 'even', 'still', 'much', 'many',
    'one', 'two', 'first', 'last',
    'get', 'got', 'getting', 'make', 'made', 'making',
    'thing', 'things', 'something', 'anything', 'nothing', 'everything',
    'way', 'time', 'times',
    'gonna', 'wanna', 'gotta', 'kinda', 'yeah', 'ok', 'okay',
    'que', 'eu', 'lol', 'haha', 'well well',
    'quote', 'quotes', 'comment', 'comments',
    'good ever', 'wrong ever', 'best ever', 'worst ever',
    'ever seen', 'ever see', 'every single',
}

stop_words_all = list(
    set(stopwords.words('turkish')) | TR_EXTRA | TR_DOMAIN |
    set(stopwords.words('english')) | EN_DOMAIN
)
print(f'Birleşik stopword sayısı: {len(stop_words_all)}')

Birleşik stopword sayısı: 545


In [3]:
from sentence_transformers import SentenceTransformer

EMB_MODEL = 'paraphrase-multilingual-MiniLM-L12-v2'
EMB_PATH = Path('processed/embeddings.npy')

embedder = SentenceTransformer(EMB_MODEL)

if EMB_PATH.exists() and len(np.load(EMB_PATH, mmap_mode='r')) == len(df):
    embeddings = np.load(EMB_PATH)
    print(f'Embedding önbellekten yüklendi: {embeddings.shape}')
else:
    print(f'{len(df):,} yorum embed ediliyor (CPU ~25-35 dk)...')
    embeddings = embedder.encode(df['review_text'].astype(str).tolist(),
                                  show_progress_bar=True, batch_size=64,
                                  convert_to_numpy=True)
    np.save(EMB_PATH, embeddings)
    print(f'Kaydedildi: {EMB_PATH}')

/Users/melihyelman/Desktop/bitirme/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9075.62it/s]


Embedding önbellekten yüklendi: (382977, 384)


In [4]:
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
from umap import UMAP
from hdbscan import HDBSCAN

umap_model = UMAP(n_neighbors=30, n_components=5, min_dist=0.0,
                  metric='cosine', random_state=42, low_memory=True)
hdbscan_model = HDBSCAN(min_cluster_size=200, min_samples=10,
                        metric='euclidean', cluster_selection_method='eom',
                        prediction_data=True)
vectorizer_model = CountVectorizer(stop_words=stop_words_all,
                                    min_df=50, max_df=0.95, ngram_range=(1, 2))

topic_model = BERTopic(
    embedding_model=embedder,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    language='multilingual',
    calculate_probabilities=False,
    verbose=True,
)

cleaned_docs = df['cleaned_text'].astype(str).tolist()
topics, _ = topic_model.fit_transform(cleaned_docs, embeddings)
print(f'İlk kümeleme — konu sayısı: {len([t for t in set(topics) if t != -1])}, outlier: {sum(1 for t in topics if t == -1):,}')

2026-06-02 20:22:13,084 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-06-02 20:30:10,161 - BERTopic - Dimensionality - Completed ✓
2026-06-02 20:30:10,167 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-06-02 20:30:31,415 - BERTopic - Cluster - Completed ✓
2026-06-02 20:30:31,445 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-06-02 20:30:57,726 - BERTopic - Representation - Completed ✓


İlk kümeleme — konu sayısı: 239, outlier: 192,631


In [ ]:
new_topics = topic_model.reduce_outliers(cleaned_docs, topics, strategy='c-tf-idf', threshold=0.0)
topic_model.update_topics(cleaned_docs, topics=new_topics, vectorizer_model=vectorizer_model)
print(f'Outlier indirgeme sonrası outlier: {sum(1 for t in new_topics if t == -1):,}')

2026-06-02 20:31:12,579 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


Outlier indirgeme sonrası outlier: 584


In [ ]:
topic_model.reduce_topics(cleaned_docs, nr_topics=80)
df['topic'] = topic_model.topics_
n_final = len([t for t in set(df['topic']) if t != -1])
print(f'Son konu sayısı: {n_final}')

2026-06-02 20:31:39,624 - BERTopic - Topic reduction - Reducing number of topics
2026-06-02 20:31:39,762 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-06-02 20:32:06,142 - BERTopic - Representation - Completed ✓
2026-06-02 20:32:06,164 - BERTopic - Topic reduction - Reduced number of topics from 240 to 80


Son konu sayısı: 79


In [7]:
topic_info = topic_model.get_topic_info()
print(topic_info.head(25).to_string(index=False, max_colwidth=110))

 Topic  Count                                     Name                                                                                          Representation                                                                                            Representative_Docs
    -1    584          -1_jacob_bore_surekli_bil insan [jacob, bore, surekli, bil insan, devam güzel, ayri, yuzden, man know, güzel güzel, kesinlikle tavsiye] [bore miss fun bore miss fun, thinke bore name day mfs literally use call maximus decimus meridius, jacob t...
     0  55531                 0_saat_ağla_aksiyon_ödül                                [saat, ağla, aksiyon, ödül, horror, sıkıcı, dakika, korku, seri, sistem] [ülke sınıf ayrım ayaküstü tartış farhadi muhteşem müthiş özgüven iran ınkini araştır irdele ön koy merkez ...
     1  50707        1_jack_kubrick_eastwood_hitchcock                         [jack, kubrick, eastwood, hitchcock, clint, nicholson, nolan, bill, kane, brad] [apartment often regarde great 

In [8]:
out_dir = Path('processed')
df[['imdb_id', 'title', 'source', 'language', 'review_text', 'cleaned_text', 'topic']].to_csv(
    out_dir / 'bertopic_tam_assignments.csv', index=False, encoding='utf-8-sig')
topic_info.to_csv(out_dir / 'bertopic_tam_topics.csv', index=False, encoding='utf-8-sig')
topic_model.save(str(out_dir / 'bertopic_tam_model'),
                 serialization='safetensors', save_ctfidf=True, save_embedding_model=False)
print('Kaydedildi: bertopic_tam_assignments.csv, bertopic_tam_topics.csv, bertopic_tam_model/')

2026-06-02 20:32:14,160 - BERTopic - WARNING: You are saving a BERTopic model without explicitly defining an embedding model.If you are using a sentence-transformers model or a HuggingFace model supportedby sentence-transformers, please save the model by using a pointer towards that model.For example, `save_embedding_model='sentence-transformers/all-mpnet-base-v2'`


Kaydedildi: bertopic_tam_assignments.csv, bertopic_tam_topics.csv, bertopic_tam_model/
